# **Exploracion de la ocupacion hotelera de Solola**

Grupo 3 - CC3084 Data Science.

## **1. Preparacion de datos**

### **1.1. Carga de la fuente**

El archivo `indice_ocupacion_mensual_2023_2025_1.csv` contiene el indice mensual de ocupacion hotelera de los departamentos de Guatemala, entre 2023 y 2025. Para este proyecto se seleccionan los registros de Solola, departamento al que pertenecen los municipios del Lago de Atitlan.

La informacion corresponde al departamento completo, por lo que no permite conocer la ocupacion de cada municipio del lago. En esta primera etapa, el objetivo es identificar con que datos trabajamos y que representa cada variable.

In [1]:
from pathlib import Path
import pandas as pd

raiz_proyecto = Path.cwd()
if not (raiz_proyecto / "data" / "raw").is_dir():
    raiz_proyecto = raiz_proyecto.parent

ruta_datos = raiz_proyecto / "data" / "raw" / "indice_ocupacion_mensual_2023_2025_1.csv"
datos = pd.read_csv(ruta_datos, dtype=str, keep_default_na=False, encoding="utf-8")
datos_solola = datos.loc[datos["departamento"].eq("Sololá")].copy()

### **1.2. Primeras cinco filas**

Se muestran las primeras cinco observaciones de Solola. Cada fila corresponde a un mes del departamento.

In [2]:
datos_solola.head()

,anio,mes,departamento,indice_ocupacion
216,2023,1,Sololá,0.79
217,2023,2,Sololá,0.94
218,2023,3,Sololá,0.85
219,2023,4,Sololá,0.85
220,2023,5,Sololá,0.93


## **2. Descripcion de las variables**

- **anio:** anio al que corresponde el registro de ocupacion hotelera; variable temporal de tipo cuantitativa discreta.

- **mes:** numero del mes al que corresponde el registro; variable cualitativa ordinal de caracter ciclico.

- **departamento:** departamento al que corresponde el registro de ocupacion hotelera. En la seleccion de este notebook aparece Solola; variable cualitativa nominal.

- **indice_ocupacion:** indice mensual de ocupacion hotelera reportado por la fuente, expresado en decimales, por ejemplo, `0.79`; variable cuantitativa continua. Representa una medida de ocupacion, no una cantidad de visitantes.

## **3. Revision de calidad de los datos**

### **3.1. Estructura, alcance y faltantes**

Primero se revisa la estructura del archivo completo y despues los registros seleccionados de Solola. Se buscan celdas vacias, marcadores de ausencia y espacios externos, sin modificar los textos de origen. Los calculos de diagnostico se realizan en una copia.

In [3]:
import hashlib
import numpy as np

huella_inicial = hashlib.sha256(ruta_datos.read_bytes()).hexdigest()
datos_originales = datos.copy(deep=True)
COLUMNAS_ESPERADAS = ["anio", "mes", "departamento", "indice_ocupacion"]
MARCADORES_AUSENCIA = {"", "null", "none", "nan", "na", "n/a", "s/d"}
assert list(datos.columns) == COLUMNAS_ESPERADAS
textos = datos.apply(lambda columna: columna.astype("string").str.strip())
faltantes = textos.apply(lambda columna: columna.isna() | columna.str.casefold().isin(MARCADORES_AUSENCIA))
seleccion_solola = datos["departamento"].eq("Sololá")
display(pd.DataFrame([
    {"alcance":"archivo completo", "filas":len(datos), "departamentos":datos["departamento"].nunique()},
    {"alcance":"Solola", "filas":int(seleccion_solola.sum()), "departamentos":datos_solola["departamento"].nunique()},
]))
pd.DataFrame({
    "faltantes_archivo":faltantes.sum(),
    "faltantes_solola":faltantes.loc[seleccion_solola].sum(),
    "espacios_externos":datos.ne(textos).sum(),
    "valores_distintos_solola":datos_solola.nunique(dropna=False),
})

,alcance,filas,departamentos
0,archivo completo,792,22
1,Solola,36,1


,faltantes_archivo,faltantes_solola,espacios_externos,valores_distintos_solola
anio,0,0,0,3
mes,0,0,0,12
departamento,0,0,0,1
indice_ocupacion,0,0,0,27


### **3.2. Tipos, fechas y rango del indice**

Se comprueba que anio y mes sean enteros validos, y que el indice sea numerico y finito. La representacion observada es una proporcion entre cero y uno; su presentacion porcentual se obtiene multiplicando por cien, sin cambiar el valor almacenado. Tambien se identifican extremos y ceros, sin considerarlos errores automaticamente.

In [4]:
revision = datos.copy(deep=True)
for columna in ["anio", "mes", "indice_ocupacion"]:
    revision[columna] = pd.to_numeric(textos[columna].mask(faltantes[columna]), errors="coerce")
revision["fecha"] = pd.to_datetime(dict(year=revision["anio"], month=revision["mes"], day=1), errors="coerce")
solola_revision = revision.loc[seleccion_solola].copy()
incidencias = pd.DataFrame({
    "anio_no_numerico_o_no_finito": ~np.isfinite(revision["anio"]),
    "anio_no_entero": revision["anio"].mod(1).ne(0),
    "anio_fuera_de_cobertura": ~revision["anio"].between(2023,2025),
    "mes_no_numerico_o_no_finito": ~np.isfinite(revision["mes"]),
    "mes_no_entero_o_fuera_de_rango": revision["mes"].mod(1).ne(0) | ~revision["mes"].between(1,12),
    "fecha_invalida": revision["fecha"].isna(),
    "indice_no_numerico_o_no_finito": ~np.isfinite(revision["indice_ocupacion"]),
    "indice_fuera_de_0_a_1": ~revision["indice_ocupacion"].between(0,1),
})
display(pd.DataFrame({"archivo_completo":incidencias.sum(), "solola":incidencias.loc[seleccion_solola].sum()}))
display(revision.loc[incidencias.any(axis=1), COLUMNAS_ESPERADAS])
pd.DataFrame({
    "archivo_completo":revision["indice_ocupacion"].agg(["min","max"]),
    "solola":solola_revision["indice_ocupacion"].agg(["min","max"]),
})

,archivo_completo,solola
anio_no_numerico_o_no_finito,0,0
anio_no_entero,0,0
anio_fuera_de_cobertura,0,0
mes_no_numerico_o_no_finito,0,0
mes_no_entero_o_fuera_de_rango,0,0
fecha_invalida,0,0
indice_no_numerico_o_no_finito,0,0
indice_fuera_de_0_a_1,0,0


,anio,mes,departamento,indice_ocupacion


,archivo_completo,solola
min,0.15,0.20
max,0.98,0.96


### **3.3. Duplicados y cobertura mensual**

La clave de cada observacion es departamento, anio y mes. Dos meses con el mismo indice no son duplicados, porque pueden representar observaciones diferentes con igual ocupacion. Se contrasta la tabla con la rejilla esperada de 22 departamentos y 36 meses, para identificar tanto ausencias como combinaciones repetidas.

In [5]:
clave = ["departamento", "anio", "mes"]
display(pd.Series({
    "filas_exactamente_duplicadas":int(datos.duplicated().sum()),
    "repeticiones_adicionales_clave_textual":int(datos.duplicated(clave).sum()),
    "repeticiones_adicionales_clave_tipificada":int(revision.duplicated(clave).sum()),
    "repeticiones_clave_solola":int(solola_revision.duplicated(clave).sum()),
}).to_frame("cantidad"))
calendario = pd.date_range("2023-01-01", "2025-12-01", freq="MS")
rejilla = pd.MultiIndex.from_product([sorted(datos["departamento"].unique()), calendario], names=["departamento","fecha"])
observadas = pd.MultiIndex.from_frame(revision[["departamento","fecha"]])
display(rejilla.difference(observadas).to_frame(index=False).rename(columns={"fecha":"mes_sin_registro"}))
cobertura = revision.groupby(["departamento","anio"])["mes"].nunique().unstack("anio")
display(cobertura)
pd.DataFrame({
    "meses_esperados":[len(calendario)],
    "meses_observados_solola":[solola_revision["fecha"].nunique()],
    "meses_ausentes_solola":[len(calendario.difference(solola_revision["fecha"]))],
    "orden_temporal_solola":[solola_revision["fecha"].is_monotonic_increasing],
})

,cantidad
filas_exactamente_duplicadas,0
repeticiones_adicionales_clave_textual,0
repeticiones_adicionales_clave_tipificada,0
repeticiones_clave_solola,0


,departamento,mes_sin_registro


anio,2023,2024,2025
departamento,,,
Alta Verapaz,12,12,12
Baja Verapaz,12,12,12
Chimaltenango,12,12,12
Chiquimula,12,12,12
El Progreso,12,12,12
Escuintla,12,12,12
Guatemala,12,12,12
Huehuetenango,12,12,12
Izabal,12,12,12


,meses_esperados,meses_observados_solola,meses_ausentes_solola,orden_temporal_solola
0,36,36,0,True


### **3.4. Correspondencia con los boletines de ocupacion**

Se contrastan los 36 porcentajes mensuales de Solola con las tablas de los boletines incluidos en `data/raw`. Las referencias siguientes se transcribieron de las filas mensuales de esos documentos; el codigo comprueba su correspondencia mes a mes con el CSV. No se utiliza el total anual del boletin como si fuera un mes adicional.

- [Boletin 2023](../data/raw/BOLETIN%20OCUPACION%20HOTELERA%20ENERO%20A%20DICIEMBRE%202023.pdf): pagina 13 del PDF, pagina impresa 10. La fila se rotula `Solola, Panajachel`.
- [Boletin 2024](../data/raw/BOLETIN%20OCUPACION%20HOTELERA%20ENERO%20A%20DICIEMBRE%202024.pdf): pagina 10 del PDF, pagina impresa 10. La fila se rotula `Solola, Panajachel`.
- [Boletin 2025](../data/raw/BOLETIN%20OCUPACION%20HOTELERA%20ENERO-%20DICIEMBRE%202025.pdf): pagina 10 del PDF, pagina impresa 6. La fila se rotula `Solola`.

El CSV utiliza `Solola` en los tres anos. La diferencia del rotulo en 2023 y 2024 se documenta como una limitacion territorial: la coincidencia numerica no demuestra que todos los municipios o establecimientos del departamento tengan cobertura equivalente.

In [6]:
# Referencia transcrita de los PDF, en porcentajes y en orden enero-diciembre.
PORCENTAJES_BOLETINES = {2023: [79, 94, 85, 85, 93, 57, 62, 78, 64, 52, 20, 73], 2024: [91, 83, 84, 96, 69, 58, 51, 61, 57, 36, 46, 70], 2025: [83, 81, 74, 82, 84, 54, 57, 64, 64, 48, 61, 96]}

referencia_boletines = pd.DataFrame([
    {"anio":anio,"mes":mes,"porcentaje_boletin":valor}
    for anio, valores in PORCENTAJES_BOLETINES.items()
    for mes, valor in enumerate(valores, start=1)
])
contraste_boletines = solola_revision.merge(referencia_boletines, on=["anio","mes"], how="outer", validate="one_to_one", indicator=True)
contraste_boletines["porcentaje_csv"] = contraste_boletines["indice_ocupacion"] * 100
contraste_boletines["coincide"] = contraste_boletines["_merge"].eq("both") & np.isclose(contraste_boletines["porcentaje_csv"],contraste_boletines["porcentaje_boletin"], rtol=0, atol=1e-8)
display(contraste_boletines.groupby("anio").agg(meses=("mes","size"), coincidencias=("coincide","sum")))
contraste_boletines[["anio","mes","porcentaje_csv","porcentaje_boletin","coincide"]]

,meses,coincidencias
anio,,
2023,12,12
2024,12,12
2025,12,12


,anio,mes,porcentaje_csv,porcentaje_boletin,coincide
0,2023,1,79.0,79,True
1,2023,2,94.0,94,True
2,2023,3,85.0,85,True
3,2023,4,85.0,85,True
4,2023,5,93.0,93,True
5,2023,6,57.0,57,True
6,2023,7,62.0,62,True
7,2023,8,78.0,78,True
8,2023,9,64.0,64,True
9,2023,10,52.0,52,True


In [7]:
# Huellas de las referencias consultadas para identificar su version.
referencias_documentales = [{'anio': 2023, 'archivo': 'BOLETIN OCUPACION HOTELERA ENERO A DICIEMBRE 2023.pdf', 'pagina_pdf': 13, 'sha256': '74b861c13691531fbc0aa904efc1f7625fb15ca2039a300758711cf65a0b663f'}, {'anio': 2024, 'archivo': 'BOLETIN OCUPACION HOTELERA ENERO A DICIEMBRE 2024.pdf', 'pagina_pdf': 10, 'sha256': '3dbd7d51717532282f32ebd1f6f2a92c9f3747fc23db94159a49f4f93b322424'}, {'anio': 2025, 'archivo': 'BOLETIN OCUPACION HOTELERA ENERO- DICIEMBRE 2025.pdf', 'pagina_pdf': 10, 'sha256': 'd942c45e39520c8395b74b5e5774976e4ad634f99eead6582bba4625598e30c5'}]

for referencia in referencias_documentales:
    assert hashlib.sha256((raiz_proyecto / "data" / "raw" / referencia["archivo"]).read_bytes()).hexdigest() == referencia["sha256"]
pd.DataFrame(referencias_documentales)

,anio,archivo,pagina_pdf,sha256
0,2023,BOLETIN OCUPACION HOTELERA ENERO A DICIEMBRE 2...,13,74b861c13691531fbc0aa904efc1f7625fb15ca2039a30...
1,2024,BOLETIN OCUPACION HOTELERA ENERO A DICIEMBRE 2...,10,3dbd7d51717532282f32ebd1f6f2a92c9f3747fc23db94...
2,2025,BOLETIN OCUPACION HOTELERA ENERO- DICIEMBRE 20...,10,d942c45e39520c8395b74b5e5774976e4ad634f99eead6...


### **3.5. Valores extremos y cambios entre meses**

Se muestran los meses de menor y mayor ocupacion y los cambios mensuales de al menos 25 puntos porcentuales. Este umbral solo ayuda a localizar movimientos para inspeccion; no define errores ni activa una correccion. Las diferencias se calculan en puntos porcentuales, no como variacion porcentual relativa.

In [8]:
serie_revision = solola_revision.sort_values("fecha").set_index("fecha")["indice_ocupacion"]
cambios = serie_revision.rename("indice_ocupacion").to_frame()
cambios["cambio_puntos_porcentuales"] = serie_revision.diff()*100
extremos = solola_revision.loc[
    solola_revision["indice_ocupacion"].eq(serie_revision.min()) | solola_revision["indice_ocupacion"].eq(serie_revision.max()),
    ["anio","mes","departamento","indice_ocupacion"],
]
display(extremos)
display(cambios.loc[cambios["cambio_puntos_porcentuales"].abs().ge(25)])
contraste_boletines.loc[contraste_boletines["indice_ocupacion"].eq(0.2),["anio","mes","porcentaje_csv","porcentaje_boletin","coincide"]]

,anio,mes,departamento,indice_ocupacion
226,2023,11,Sololá,0.2
483,2024,4,Sololá,0.96
755,2025,12,Sololá,0.96


,indice_ocupacion,cambio_puntos_porcentuales
fecha,,
2023-06-01,0.57,-36.0
2023-11-01,0.2,-32.0
2023-12-01,0.73,53.0
2024-05-01,0.69,-27.0
2025-06-01,0.54,-30.0
2025-12-01,0.96,35.0


,anio,mes,porcentaje_csv,porcentaje_boletin,coincide
10,2023,11,20.0,20,True


### **3.6. Alcance del cruce con movilidad y conclusiones**

La serie de Solola tiene 36 observaciones, de enero de 2023 a diciembre de 2025, con doce meses por ano. No se encontraron faltantes, duplicados de la clave, valores no numericos ni indices fuera del intervalo de cero a uno. El archivo completo tambien cumple estos controles estructurales y tiene 792 filas, correspondientes a 22 departamentos por 36 meses.

Los 36 indices de Solola coinciden con los porcentajes de los boletines. El minimo es 0.20 en noviembre de 2023 y el maximo es 0.96 en abril de 2024 y diciembre de 2025. El 20 % de noviembre de 2023 puede llamar la atencion, pero tambien aparece en el boletin, por lo que no hay evidencia para reemplazarlo como si fuera un error de captura. Esta concordancia respalda la transcripcion, sin demostrar por si sola la precision del levantamiento original.

El indice viene calculado y el CSV no incluye habitaciones ocupadas ni habitaciones disponibles para reconstruirlo. Por eso, no se sumaran indices mensuales para obtener ocupacion anual ni se presentara una media simple como si reprodujera el indice anual institucional. La oferta anual de alojamiento tampoco sustituye automaticamente el denominador mensual de la fuente.

Para cruzar ocupacion con la movilidad observada de Atitlan, la ventana coincidente es septiembre-diciembre de 2025. La ausencia de ocupacion de 2026 corresponde al alcance de esta fuente, no a ceros que deban rellenarse. Asimismo, el indice hotelero no representa la proporcion de todos los visitantes del lago que se hospedaron en un hotel.

La columna `departamento` permanece constante despues del filtro de Solola. Su conservacion o retiro y la conversion definitiva de los tipos se discutiran en la seccion de preprocesamiento; aqui solo se documenta la calidad encontrada.

In [9]:
# Comprobacion de la ventana compartida y de la integridad de la fuente.
ventana_coincidente = solola_revision.loc[solola_revision["fecha"].between("2025-09-01","2025-12-01"), ["fecha","indice_ocupacion"]]
display(ventana_coincidente)
assert datos.equals(datos_originales)
assert hashlib.sha256(ruta_datos.read_bytes()).hexdigest() == huella_inicial
pd.Series({"tabla_original_intacta":True,"archivo_csv_intacto":True,"meses_concordantes_boletines":int(contraste_boletines["coincide"].sum())},name="verificacion")

,fecha,indice_ocupacion
752,2025-09-01,0.64
753,2025-10-01,0.48
754,2025-11-01,0.61
755,2025-12-01,0.96


tabla_original_intacta          True
archivo_csv_intacto             True
meses_concordantes_boletines      36
Name: verificacion, dtype: object

## **4. Criterios para la etapa de preprocesamiento**

### **4.1. Seleccion territorial y columnas de salida**

Se cargara el CSV original y se seleccionaran los registros cuyo departamento sea `Sololá`, conservando sus 36 observaciones mensuales de 2023 a 2025. El archivo de origen no se modificara. La salida sera exclusiva de Solola y ese alcance quedara identificado en el nombre del archivo y en su documentacion, junto con la nota territorial de los boletines.

| Columna de salida | Tratamiento previsto |
|---|---|
| `fecha` | Fecha mensual derivada de anio y mes, utilizando el primer dia del mes. Ese dia identifica el periodo y no una sola jornada. |
| `indice_ocupacion` | Numerico entre cero y uno; se conserva su escala y precision para los calculos. |

La tabla procesada tendra solamente estas dos columnas. Despues de validar y construir `fecha`, se retiraran `anio` y `mes`, porque se pueden obtener de ella. Tambien se retirara `departamento`, porque es constante en esta seleccion exclusiva de Solola. Las cuatro columnas originales seguiran disponibles en el CSV de origen.

### **4.2. Tratamiento de cantidades, faltantes y duplicados**

No se identificaron valores que requieran una reparacion de contenido. Se convertiran los tipos y se validaran las fechas, conservando los valores del indice tal como aparecen en el CSV. Para presentarlo en el tablero se utilizara formato porcentual: por ejemplo, 0.79 se mostrara como 79 %, sin guardar 79 en la misma columna ni crear otra medida independiente.

Se conservaran los extremos y los cambios mensuales, incluido 0.20 en noviembre de 2023, porque coinciden con los boletines. No se aplicaran recortes, suavizados, redondeos ni imputaciones como parte de la limpieza. Tampoco se eliminaran meses por repetir un mismo indice.

En la salida exclusiva de Solola, la clave sera `fecha`. Antes de retirar departamento se verificara tambien la unicidad de departamento y fecha en la seleccion original. La version actual no tiene duplicados; si una nueva version presenta la misma clave mas de una vez, se detendra la consolidacion para revisar los registros, sin eliminar uno arbitrariamente. Los vacios o marcadores de ausencia se representaran como nulos, sin convertirlos en cero ni descartar automaticamente sus filas. Un indice no numerico, no finito o fuera de cero a uno se registrara como incidencia y se rechazara a nulo, conservando el valor original en la evidencia. Una fecha invalida o la falta de un mes requerido impedira certificar la serie como completa.

### **4.3. Alcance territorial y cruces con otras fuentes**

La salida se identificara como ocupacion hotelera reportada para Solola, manteniendo la nota de que los boletines de 2023 y 2024 rotulan la fila como `Solola, Panajachel`. Esa diferencia no se corregira cambiando el departamento a Panajachel, ni se explicara mediante una supuesta mayor afluencia: el rotulo no permite demostrar por si solo la cobertura de cada municipio.

El indice no se asignara a cada municipio del lago como si existieran mediciones municipales. Tampoco se interpretara como la proporcion de todos los visitantes de Atitlan que se hospedaron en hoteles. El cruce con movilidad observada se limitara a septiembre-diciembre de 2025, que es la ventana coincidente de las fuentes revisadas.

La ausencia de ocupacion en 2026 se mantendra como limite de cobertura, sin rellenarla con cero ni extender la serie como parte de la limpieza. El ejercicio de estimar enero-febrero de 2026 corresponde a la movilidad mensual, no a esta serie de ocupacion. No se sumaran los indices para calcular ocupacion anual ni se presentara una media simple como si fuera el indice anual institucional, porque aqui no se dispone de sus denominadores mensuales.

### **4.4. Resultados verificados y controles de salida**

La revision de calidad de la seccion 3 ya confirmo que los 36 registros de Solola tienen anios y meses validos, fechas mensuales consecutivas, clave unica e indices numericos, finitos y dentro de cero a uno. Tambien se verifico la correspondencia de los 36 valores con los boletines. Estas comprobaciones se ejecutaron sobre una copia de diagnostico y respaldan los criterios de preprocesamiento; no son tareas pendientes para determinar si la fuente sirve.

Cuando se implemente el pipeline se repetiran los controles sobre la salida, verificando las dos columnas previstas, los tipos y la unicidad de fecha. Antes de retirar las columnas originales se comprobaran el departamento seleccionado y la correspondencia de anio y mes con la fecha derivada. Esta segunda validacion comprobara que las transformaciones no introdujeron errores y que la fuente no cambio respecto de la version revisada. Para esta version se esperan 36 meses consecutivos, desde enero de 2023 hasta diciembre de 2025, sin faltantes y con indices finitos entre cero y uno. Tambien se comprobara que los valores del indice y la cantidad de observaciones no cambien respecto de la seleccion original.

**Estos criterios documentan el preprocesamiento acordado. Las transformaciones persistentes y la generacion del archivo procesado se realizaran en la etapa de pipeline; no se aplican en este notebook.**
